In [ ]:
import pathlib

In [ ]:
import downstream
import imageio
import marimo as mo
import numpy as np
import polars as pl
from tqdm.auto import tqdm
from watermark import watermark

from pylib.munge import addend_groups, squeeze_int, union_upsample

In [ ]:
mo.md(
    f"""
```Text
{watermark(
    current_date=True,
    iso8601=True,
    machine=True,
    updated=True,
    python=True,
    iversions=True,
    globals_=globals(),
)}
```
"""
)

```Text
Last updated: 2026-11-20T16:56:29.648868+00:00

Python implementation: CPython
Python version       : 3.10.12
IPython version      : 7.31.1

Compiler    : GCC 11.4.0
OS          : Linux
Release     : 6.8.0-1070-azure
Machine     : x86_64
Processor   : x86_64
CPU cores   : 4
Architecture: 64bit

marimo    : 0.23.2
numpy     : 2.1.2
imageio   : 2.37.0
polars    : 1.40.1
downstream: 1.25.0

```

# 2026-11-19-alleleevoepi-fullgrid-p2p-traitlog

Animates strains of each PE's sampled host (0th population member)
over time, from the trait logger of wse-async-ga workflow
`2026-11-19-alleleevoepi-fullgrid-p2p-traitlog` (genome
`genome_alleleevoepi_2026_11_19`, 2 sites).

The trait logger multiplexes three single-bit records per PE
(infected, site 0 allele, site 1 allele); downstream demultiplexes
them by the `dstream_storage_multiplex` column.

Colors: black, uninfected; white, wildtype; blue, site 0 mutant; red,
site 1 mutant; purple, double mutant.

In [ ]:
nbname = pathlib.Path(__file__).stem
outdir = pathlib.Path("outdata") / nbname
outdir.mkdir(parents=True, exist_ok=True)
outdir

PosixPath('outdata/2026-11-19-alleleevoepi-fullgrid-p2p-traitlog')

## Get Data

In [ ]:
# trait logger records, one OSF file per run
osf_ids = ["hu2w5", "cxkqu", "bezac", "dtc5w", "v5hjw"]

config_columns = [
    "dispatchSlug",
    "globalSeed",
    "nRow",
    "nCol",
    "nRowSubgrid",
    "nColSubgrid",
    "tilePopSize",
    "tournSize",
    "nCycle",
    "evoepi_waning_rate",
    "evoepi_seed_pe_stride",
    "ASYNC_GA_P2P_SEND_PROBABILITY",
    "dstream_storage_multiplex",
]

## Unpack Data

In [ ]:
def unpack_strain_timeseries(df: pl.DataFrame) -> pl.DataFrame:

    # sample every 64th tile
    df = df.filter((pl.col("col") % 8 == 0) & (pl.col("row") % 8 == 0))

    # unpack dstream data items from logger hex strings, demultiplexing
    # infected, site 0 allele, and site 1 allele records
    dfu = downstream.dataframe.unpack_data_packed(df)
    df = downstream.dataframe.explode_lookup_unpacked(
        dfu.with_columns(
            # fast forward loggers to at least dstream_S entries
            # (i.e., buffer filled)
            # these will be semantically-correct zero values
            dstream_T=pl.max_horizontal(
                pl.col("dstream_T"), pl.col("dstream_S")
            ),
        ),
        value_type="uint8",
    )

    # join metadata onto unpacked data
    df = df.join(
        dfu.select("dstream_data_id", "col", "row", "tile").unique(
            "dstream_data_id",
        ),
        on="dstream_data_id",
    )

    # apply dilation to dstream_Tbar, then one column per record
    df = (
        df.with_columns(
            dstream_Tbar=pl.col("dstream_Tbar")
            * pl.col("dstream_T_dilation"),
        )
        .pivot(
            on="dstream_storage_multiplex_id",
            index=["tile", "row", "col", "dstream_Tbar"],
            values="dstream_value",
        )
        .rename({"0": "infected", "1": "site0", "2": "site1"})
    )

    # addend time series groups with final uninfected entry, after
    # logger's last seen time (i.e., last time any record was set)
    df = addend_groups(
        df,
        group_by="tile",
        aggs=(
            pl.col("col").unique().pipe(squeeze_int),
            pl.col("row").unique().pipe(squeeze_int),
            (pl.col("dstream_Tbar") + 1).max().alias("dstream_Tbar"),
            pl.lit(0).alias("infected"),
            pl.lit(0).alias("site0"),
            pl.lit(0).alias("site1"),
        ),
        inner_only=True,
    )

    # upsample to fill in dstream_Tbar dropped by some loggers
    df = union_upsample(
        df,
        upsample="dstream_Tbar",
        group_by="tile",
        fill_null_ops=[
            (pl.all(), {"strategy": "forward"}),
            (pl.all(), {"strategy": "backward"}),
        ],
    )
    assert len(df["dstream_Tbar"].value_counts()["count"].unique()) == 1

    # 0: uninfected, 1: wildtype, 2: site 0 mutant, 3: site 1 mutant,
    # 4: double mutant
    return df.with_columns(
        strain=pl.col("infected")
        * (1 + pl.col("site0") + 2 * pl.col("site1")),
    )

In [ ]:
# read and unpack one run at a time, keeping only unpacked strains
_configs, dfs = [], {}
for _osf_id in tqdm(osf_ids):
    _df = pl.read_parquet(
        f"https://osf.io/{_osf_id}/download",
    ).with_row_index("dstream_data_id")
    _configs.append(
        _df.select(config_columns)
        .unique()
        .with_columns(osf_id=pl.lit(_osf_id)),
    )
    dfs[_osf_id] = unpack_strain_timeseries(_df)
    del _df

configs = pl.concat(_configs)
assert len(configs) == len(osf_ids)
configs

  0%|          | 0/5 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/downstream/dataframe/_impl/_check_downstream_version.py:34: UserWarning: Dataframe downstream_version column not provided
  warnings.warn(
100%|██████████| 5/5 [07:10<00:00, 86.02s/it]


dispatchSlug,globalSeed,nRow,nCol,nRowSubgrid,nColSubgrid,tilePopSize,tournSize,nCycle,evoepi_waning_rate,evoepi_seed_pe_stride,ASYNC_GA_P2P_SEND_PROBABILITY,dstream_storage_multiplex,osf_id
cat,u32,u16,u16,u16,u16,u16,f32,u32,f32,u16,f32,u8,str
"""p2p-none""",1119,1170,755,0,0,6800,2.0,50000,0.01,1170,0.002,3,"""hu2w5"""
"""p2p-pairs""",1120,1170,755,0,0,6800,2.0,50000,0.01,1170,0.002,3,"""cxkqu"""
"""p2p-pairs""",1122,1170,755,0,0,6800,2.0,50000,0.0,1170,0.002,3,"""bezac"""
"""p2p-pairs""",1121,1170,755,0,0,6800,2.0,50000,1.0,1170,0.002,3,"""dtc5w"""
"""p2p-pairs""",1123,729,729,243,243,6800,2.0,50000,0.01,243,0.002,3,"""v5hjw"""


In [ ]:
pl.concat(
    [
        _df.group_by("strain").len().with_columns(osf_id=pl.lit(_osf_id))
        for _osf_id, _df in dfs.items()
    ],
).pivot(on="strain", index="osf_id", values="len", sort_columns=True)

osf_id,0,1,2,3,4
str,u64,u64,u64,u64,u64
"""hu2w5""",13170205,296454,277251,276753,293462
"""cxkqu""",11596032,688322,672049,671489,686233
"""bezac""",25171304,92428,30486,31235,90847
"""dtc5w""",4893752,8609078,414722,368416,28157
"""v5hjw""",6918056,464688,416347,416928,459581


## Animate Data

In [ ]:
palette = np.array(
    [
        [0, 0, 0],  # uninfected: black
        [255, 255, 255],  # wildtype: white
        [0, 0, 255],  # site 0 mutant: blue
        [255, 0, 0],  # site 1 mutant: red
        [160, 32, 240],  # double mutant: purple
    ],
    dtype=np.uint8,
)

In [ ]:
for _osf_id, _df in tqdm(dfs.items()):
    _config = configs.filter(pl.col("osf_id") == _osf_id).row(0, named=True)
    _label = (
        f"p2pmode={_config['dispatchSlug'].removeprefix('p2p-')}"
        f"+subgrid={_config['nRowSubgrid']}"
        f"+waning={_config['evoepi_waning_rate']:.2f}"
        f"+seed={_config['globalSeed']}"
    )
    print(_osf_id, _label)

    _df = _df.with_columns(
        row_rank=pl.col("row").rank(method="dense").cast(pl.Int32) - 1,
        col_rank=pl.col("col").rank(method="dense").cast(pl.Int32) - 1,
    )
    _num_rows = _df["row_rank"].max() + 1
    _num_cols = _df["col_rank"].max() + 1

    _frames = []
    for _, _dff in _df.sort("dstream_Tbar").group_by(
        "dstream_Tbar", maintain_order=True
    ):
        _frame = np.zeros((_num_rows, _num_cols), dtype=np.uint8)
        _frame[_dff["row_rank"], _dff["col_rank"]] = _dff["strain"]
        _frame = np.kron(
            palette[_frame], np.ones((4, 4, 1), dtype=np.uint8)
        )
        _frames.append(_frame)

    print(len(_frames))
    _outpath = outdir / f"a=strains+{_label}+osf={_osf_id}+ext="
    imageio.mimsave(f"{_outpath}.gif", _frames, duration=0.05, loop=0)
    imageio.mimsave(
        f"{_outpath}.mp4", _frames, fps=30, macro_block_size=1
    )

  0%|          | 0/5 [00:00<?, ?it/s]

hu2w5 p2pmode=none+subgrid=0+waning=0.01+seed=1119
1025


 20%|██        | 1/5 [00:29<01:57, 29.48s/it]

cxkqu p2pmode=pairs+subgrid=0+waning=0.01+seed=1120
1025


 40%|████      | 2/5 [01:00<01:31, 30.65s/it]

bezac p2pmode=pairs+subgrid=0+waning=0.00+seed=1122
1820


 60%|██████    | 3/5 [01:43<01:12, 36.23s/it]

dtc5w p2pmode=pairs+subgrid=0+waning=1.00+seed=1121
1025


 80%|████████  | 4/5 [02:14<00:34, 34.14s/it]

v5hjw p2pmode=pairs+subgrid=243+waning=0.01+seed=1123
1025


100%|██████████| 5/5 [02:35<00:00, 31.01s/it]
